In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import cv2
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
import numpy as np
import seaborn as sns
!pip install --quiet shap
import shap
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from keras.applications.vgg16 import VGG16,preprocess_input

In [ ]:
# Install and import LIME
!pip install lime

from lime import lime_image
from skimage.segmentation import mark_boundaries

In [ ]:
# Initialize lists
train_Ax_images = []
train_Ax_labels = []
shape = (128, 128)  # Resizing to 224x224
train_Ax_path = 'ax_AD_CN_MCI'

# Loop through the images in the directory
for filename in os.listdir(train_Ax_path):
    if filename.split('.')[-1] == 'jpg':  # Ensure it's a .jpg file
        img = cv2.imread(os.path.join(train_Ax_path, filename), cv2.IMREAD_COLOR)  # Load in RGB
        # Extract label from filename and store
        train_Ax_labels.append(filename.split('_')[0])
        # Resize image to the desired shape
        img = cv2.resize(img, shape)
        train_Ax_images.append(img)

# Convert labels to One Hot encoded sparse matrix
train_Ax_labels = pd.get_dummies(train_Ax_labels).values

# Convert train_images to a NumPy array
train_Ax_images = np.array(train_Ax_images, dtype=np.float32)

# Normalize image pixel values to [0, 1]
train_Ax_images = train_Ax_images / 255.0

# Split training data into training and testing datasets
x_train1_Ax, x_test_Ax, y_train1_Ax, y_test_Ax = train_test_split(
    train_Ax_images, train_Ax_labels, test_size=0.2, shuffle=True, random_state=42
)

# Shuffle training data
x_train1_Ax, y_train1_Ax = shuffle(x_train1_Ax, y_train1_Ax, random_state=42)

# Further split training data into training and validation datasets
x_train_Ax, v_train_Ax, y_train_Ax, v_test_Ax = train_test_split(
    x_train1_Ax, y_train1_Ax, test_size=0.1, shuffle=True, random_state=42
)

# Check shapes
print("Train Labels Shape:", train_Ax_labels.shape)
print("Train Images Shape:", train_Ax_images.shape)
print("Training Data Shape:", x_train_Ax.shape, y_train_Ax.shape)
print("Testing Data Shape:", x_test_Ax.shape, y_test_Ax.shape)
print("Validation Data Shape:", v_train_Ax.shape, v_test_Ax.shape)

In [ ]:
# Initialize lists
train_Cr_images = []
train_Cr_labels = []
shape = (128, 128)  # Resizing to 224x224
train_Cr_path = 'cr_AD_CN_MCI'

# Loop through the images in the directory
for filename in os.listdir(train_Cr_path):
    if filename.split('.')[-1] == 'jpg':  # Ensure it's a .jpg file
        img = cv2.imread(os.path.join(train_Cr_path, filename), cv2.IMREAD_COLOR)  # Load in RGB
        # Extract label from filename and store
        train_Cr_labels.append(filename.split('_')[0])
        # Resize image to the desired shape
        img = cv2.resize(img, shape)
        train_Cr_images.append(img)

# Convert labels to One Hot encoded sparse matrix
train_Cr_labels = pd.get_dummies(train_Cr_labels).values

# Convert train_images to a NumPy array
train_Cr_images = np.array(train_Cr_images, dtype=np.float32)

# Normalize image pixel values to [0, 1]
train_Cr_images = train_Cr_images / 255.0

# Split training data into training and testing datasets
x_train1_Cr, x_test_Cr, y_train1_Cr, y_test_Cr = train_test_split(
    train_Cr_images, train_Cr_labels, test_size=0.2, shuffle=True, random_state=42
)

# Shuffle training data
x_train1_Cr, y_train1_Cr = shuffle(x_train1_Cr, y_train1_Cr, random_state=42)

# Further split training data into training and validation datasets
x_train_Cr, v_train_Cr, y_train_Cr, v_test_Cr = train_test_split(
    x_train1_Cr, y_train1_Cr, test_size=0.1, shuffle=True, random_state=42
)

# Check shapes
print("Train Labels Shape:", train_Cr_labels.shape)
print("Train Images Shape:", train_Cr_images.shape)
print("Training Data Shape:", x_train_Cr.shape, y_train_Cr.shape)
print("Testing Data Shape:", x_test_Cr.shape, y_test_Cr.shape)
print("Validation Data Shape:", v_train_Cr.shape, v_test_Cr.shape)

In [ ]:
# # Initialize lists
# train_Sg_images = []
# train_Sg_labels = []
# shape = (128, 128)  # Resizing to 224x224
# train_Sg_path = 'sg_AD_CN_MCI'

# # Loop through the images in the directory
# for filename in os.listdir(train_Sg_path):
#     if filename.split('.')[-1] == 'jpg':  # Ensure it's a .jpg file
#         img = cv2.imread(os.path.join(train_Sg_path, filename), cv2.IMREAD_COLOR)  # Load in RGB
#         # Extract label from filename and store
#         train_Sg_labels.append(filename.split('_')[0])
#         # Resize image to the desired shape
#         img = cv2.resize(img, shape)
#         train_Sg_images.append(img)

# # Convert labels to One Hot encoded sparse matrix
# train_Sg_labels = pd.get_dummies(train_Sg_labels).values

# # Convert train_images to a NumPy array
# train_Sg_images = np.array(train_Sg_images, dtype=np.float32)

# # Normalize image pixel values to [0, 1]
# train_Sg_images = train_Sg_images / 255.0

# # Split training data into training and testing datasets
# x_train1_Sg, x_test_Sg, y_train1_Sg, y_test_Sg = train_test_split(
#     train_Sg_images, train_Sg_labels, test_size=0.2, shuffle=True, random_state=42
# )

# # Shuffle training data
# x_train1_Sg, y_train1_Sg = shuffle(x_train1_Sg, y_train1_Sg, random_state=42)

# # Further split training data into training and validation datasets
# x_train_Sg, v_train_Sg, y_train_Sg, v_test_Sg = train_test_split(
#     x_train1_Sg, y_train1_Sg, test_size=0.1, shuffle=True, random_state=42
# )

# # Check shapes
# print("Train Labels Shape:", train_Sg_labels.shape)
# print("Train Images Shape:", train_Sg_images.shape)
# print("Training Data Shape:", x_train_Sg.shape, y_train_Sg.shape)
# print("Testing Data Shape:", x_test_Sg.shape, y_test_Sg.shape)
# print("Validation Data Shape:", v_train_Sg.shape, v_test_Sg.shape)

In [ ]:
# Concatenate AX, CR, and SG training images and labels
x_train_concatenated = np.concatenate((train_Ax_images, train_Cr_images), axis=0)    # train_Cr_images), axis=0)     
y_train_concatenated = np.concatenate((train_Ax_labels, train_Cr_labels), axis=0)    # train_Sg_labels), axis=0)     
x_test_concatenated = np.concatenate((x_test_Ax, x_test_Cr), axis=0)                 #x_test_Cr), axis=0)                  
y_test_concatenated = np.concatenate((y_test_Ax, y_test_Cr), axis=0)                 #y_test_Sg), axis=0)                  
x_val_concatenated = np.concatenate((v_train_Ax, v_train_Cr), axis=0)                #v_train_Cr), axis=0)                 
y_val_concatenated = np.concatenate((v_test_Ax, v_test_Cr), axis=0)                  #v_test_Sg), axis=0)           


In [ ]:
# Splitting Training data into train and test dataset
x_train1,x_test,y_train1,y_test=train_test_split(x_train_concatenated,y_train_concatenated,test_size=0.2,shuffle=True,random_state=42)#,stratify=train_images)

x_train1,y_train1=shuffle(x_train1, y_train1)

#Splitting Training data into train and validation dataset
x_train,v_train,y_train,v_test=train_test_split(x_train1,y_train1,test_size=0.1,shuffle=True,random_state=42)#,stratify=y_train1)

# Debugging: Print the shapes of the splits
print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")
print(f"x_val shape: {v_train.shape}, y_val shape: {v_test.shape}")

In [ ]:
# # Initialize lists
# train_Ax_images = []
# train_Ax_labels = []
# shape = (128, 128)  # Resizing to 224x224
# train_Ax_path = 'Slices_Separate_Folders_T1_weighted/ax_AD_CN_MCI'

# # Loop through the images in the directory
# for filename in os.listdir(train_Ax_path):
#     if filename.split('.')[-1] == 'jpg':  # Ensure it's a .jpg file
#         img = cv2.imread(os.path.join(train_Ax_path, filename), cv2.IMREAD_COLOR)  # Load in RGB
#         # Extract label from filename and store
#         train_Ax_labels.append(filename.split('_')[0])
#         # Resize image to the desired shape
#         img = cv2.resize(img, shape)
#         train_Ax_images.append(img)

# # Convert labels to One Hot encoded sparse matrix
# train_Ax_labels = pd.get_dummies(train_Ax_labels).values

# # Convert train_images to a NumPy array
# train_Ax_images = np.array(train_Ax_images, dtype=np.float32)

# # Normalize image pixel values to [0, 1]
# train_Ax_images = train_Ax_images / 255.0

# # Split training data into training and testing datasets
# x_train1, x_test, y_train1, y_test = train_test_split(
#     train_Ax_images, train_Ax_labels, test_size=0.2, shuffle=True, random_state=42
# )

# # Shuffle training data
# x_train1, y_train1 = shuffle(x_train1, y_train1, random_state=42)

# # Further split training data into training and validation datasets
# x_train, v_train, y_train, v_test = train_test_split(
#     x_train1, y_train1, test_size=0.1, shuffle=True, random_state=42
# )

# # Check shapes
# print("Train Labels Shape:", train_Ax_labels.shape)
# print("Train Images Shape:", train_Ax_images.shape)
# print("Training Data Shape:", x_train.shape, y_train.shape)
# print("Testing Data Shape:", x_test.shape, y_test.shape)
# print("Validation Data Shape:", v_train.shape, v_test.shape)

In [ ]:
# # Initialize lists
# train_Cr_images = []
# train_Cr_labels = []
# shape = (128, 128)  # Resizing to 224x224
# train_Cr_path = 'Slices_Separate_Folders_T1_weighted/cr_AD_CN_MCI'

# # Loop through the images in the directory
# for filename in os.listdir(train_Cr_path):
#     if filename.split('.')[-1] == 'jpg':  # Ensure it's a .jpg file
#         img = cv2.imread(os.path.join(train_Cr_path, filename), cv2.IMREAD_COLOR)  # Load in RGB
#         # Extract label from filename and store
#         train_Cr_labels.append(filename.split('_')[0])
#         # Resize image to the desired shape
#         img = cv2.resize(img, shape)
#         train_Cr_images.append(img)

# # Convert labels to One Hot encoded sparse matrix
# train_Cr_labels = pd.get_dummies(train_Cr_labels).values

# # Convert train_images to a NumPy array
# train_Cr_images = np.array(train_Cr_images, dtype=np.float32)

# # Normalize image pixel values to [0, 1]
# train_Cr_images = train_Cr_images / 255.0

# # Split training data into training and testing datasets
# x_train1, x_test, y_train1, y_test = train_test_split(
#     train_Cr_images, train_Cr_labels, test_size=0.2, shuffle=True, random_state=42
# )

# # Shuffle training data
# x_train1, y_train1 = shuffle(x_train1, y_train1, random_state=42)

# # Further split training data into training and validation datasets
# x_train, v_train, y_train, v_test = train_test_split(
#     x_train1, y_train1, test_size=0.1, shuffle=True, random_state=42
# )

# # Check shapes
# print("Train Labels Shape:", train_Cr_labels.shape)
# print("Train Images Shape:", train_Cr_images.shape)
# print("Training Data Shape:", x_train.shape, y_train.shape)
# print("Testing Data Shape:", x_test.shape, y_test.shape)
# print("Validation Data Shape:", v_train.shape, v_test.shape)

In [ ]:
# # Initialize lists
# train_Sg_images = []
# train_Sg_labels = []
# shape = (128, 128)  # Resizing to 224x224
# train_Sg_path = 'Slices_Separate_Folders_T1_weighted/sg_AD_CN_MCI'

# # Loop through the images in the directory
# for filename in os.listdir(train_Sg_path):
#     if filename.split('.')[-1] == 'jpg':  # Ensure it's a .jpg file
#         img = cv2.imread(os.path.join(train_Sg_path, filename), cv2.IMREAD_COLOR)  # Load in RGB
#         # Extract label from filename and store
#         train_Sg_labels.append(filename.split('_')[0])
#         # Resize image to the desired shape
#         img = cv2.resize(img, shape)
#         train_Sg_images.append(img)

# # Convert labels to One Hot encoded sparse matrix
# train_Sg_labels = pd.get_dummies(train_Sg_labels).values

# # Convert train_images to a NumPy array
# train_Sg_images = np.array(train_Sg_images, dtype=np.float32)

# # Normalize image pixel values to [0, 1]
# train_Sg_images = train_Sg_images / 255.0

# # Split training data into training and testing datasets
# x_train1, x_test, y_train1, y_test = train_test_split(
#     train_Sg_images, train_Sg_labels, test_size=0.2, shuffle=True, random_state=42
# )

# # Shuffle training data
# x_train1, y_train1 = shuffle(x_train1, y_train1, random_state=42)

# # Further split training data into training and validation datasets
# x_train, v_train, y_train, v_test = train_test_split(
#     x_train1, y_train1, test_size=0.1, shuffle=True, random_state=42
# )

# # Check shapes
# print("Train Labels Shape:", train_Sg_labels.shape)
# print("Train Images Shape:", train_Sg_images.shape)
# print("Training Data Shape:", x_train.shape, y_train.shape)
# print("Testing Data Shape:", x_test.shape, y_test.shape)
# print("Validation Data Shape:", v_train.shape, v_test.shape)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#import tensorflow_addons as tfa
num_classes = 3
input_shape = (128,128, 3)

In [ ]:
learning_rate = 0.001
weight_decay = 0.0001
batch_size = 256   #128  #32    #16   #256 #64
image_size = 128 # We'll resize input images to this size
patch_size = 16 # Size of the patches to be extract from the input images
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
] # Size of the transformer layers
transformer_layers = 8  #12  #16 #8
mlp_head_units = [2048, 1024]        #[4096, 2048]     #[3072, 1536]    #[2048, 1024]   #[2048, 1024][128, 64]mlp_head_units = [3072, 1536]

data_augmentation = keras.Sequential(
    [
        #layers.Rescaling(1./255),       ####
        layers.Normalization(),
        #layers.Resizing(image_size, image_size),
        layers.RandomFlip("horizontal"),  # Updated for both horizontal and vertical flips   layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),        
        layers.RandomZoom(
            height_factor=0.2, width_factor=0.2
        ),
        #layers.RandomContrast(0.3),        ####
        #layers.RandomBrightness(0.2),      ####
        #layers.GaussianNoise(0.05),        ####
    ],
    name="data_augmentation",
)
# Compute the mean and the variance of the training data for normalization.
data_augmentation.layers[0].adapt(x_train)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x) # GELU activation func
        x = layers.Dropout(dropout_rate)(x) # Dropout layer
    return x

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        # Ensure patch_size is a tuple (e.g., (16, 16))
        if isinstance(patch_size, int):
            self.patch_size = (patch_size, patch_size)
        else:
            self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patch_height, patch_width = self.patch_size

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, patch_height, patch_width, 1],
            strides=[1, patch_height, patch_width, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )

        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

In [ ]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patches) + self.position_embedding(positions)
        return encoded        

In [ ]:
def ViT_Block():
    # Create the input tensor for the full model
    x_input = layers.Input(shape=input_shape)
    augmented = data_augmentation(x_input)
    patches = Patches(patch_size)(augmented)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    x = encoded_patches
    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attention_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        x = layers.Add()([x3, x2])

    # x now contains the encoded feature representation
    model = keras.Model(inputs=x_input, outputs=x)
    return model

In [ ]:
input_shape = (128, 128, 3)

# Define ViT encoder
vit_encoder = ViT_Block()

# Create input tensor
x_input = keras.Input(shape=input_shape)

# Get encoded features
vit_features = vit_encoder(x_input)

In [ ]:
num_classes = 3 # Number of output classes
input_shape = (128, 128, 3) #(32, 32, 3) # For RGB images of size 224x224
patch_size = (16, 16) # Update patch size to ensure compatibility
#patch_size = (2, 2) # 2D patch size (height, width)
dropout_rate = 0.03 # Dropout rate
num_heads = 4 #16 #12 #8 # Number of attention heads
embed_dim = 64       #192 #96  # Embedding dimension
num_mlp = 256          #768 #384  # Size of MLP layers
qkv_bias = True # Query, key, value bias
window_size = 4 #2 #(2, 2) # 2D attention window size
shift_size = 2 #1 #(1, 1) # Shift size for attention window
image_dimension = 128 # Initial image dimension
learning_rate = 1e-3 # Learning rate
#batch_size = 32 # Batch size
num_epochs = 200 # Number of epochs
validation_split = 0.1 # Validation split
weight_decay = 0.0001 # Weight decay
label_smoothing = 0.1 # Label smoothing
# Compute the number of patches in 2D
num_patch_x = input_shape[0] // patch_size[0]
num_patch_y = input_shape[1] // patch_size[1]
num_patches = num_patch_x * num_patch_y # Total number of patches
# Display patch information
print(f"Number of patches (x, y): ({num_patch_x}, {num_patch_y})")
print(f"Total number of patches: {num_patches}")

In [ ]:
def window_partition(x, window_size):
    _, height, width, channels = x.shape
    num_patch_y = height // window_size
    num_patch_x = width // window_size
    x = tf.reshape( #ops.reshape(
        x,
        (
            -1,
            num_patch_y,
            window_size,
            num_patch_x,
            window_size,
            channels,
        ),
    )
    x = tf.transpose(x, (0, 1, 3, 2, 4, 5)) #ops.transpose(x, (0, 1, 3, 2, 4
    windows = tf.reshape(x, (-1, window_size, window_size, channels)) #ops
    return windows
    
def window_reverse(windows, window_size, height, width, channels):
    num_patch_y = height // window_size
    num_patch_x = width // window_size
    x = tf.reshape(
        windows,
        (
            -1,
            num_patch_y,
            num_patch_x,
            window_size,
            window_size,
            channels,
        ),
    )
    x = tf.transpose(x, (0, 1, 3, 2, 4, 5)) #ops.transpose(x, (0, 1, 3, 2,
    x = tf.reshape(x, (-1, height, width, channels)) #ops.reshape(x, (-1, heigh
    return x

In [ ]:
from keras import backend as K
from tensorflow.keras import layers

In [ ]:
class WindowAttention(layers.Layer):
    def __init__(
        self,
        dim,
        window_size,
        num_heads,
        qkv_bias=True,
        dropout_rate=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.window_size = window_size if isinstance(window_size, tuple) else (window_size, window_size)
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = layers.Dense(dim * 3, use_bias=qkv_bias)
        self.dropout = layers.Dropout(dropout_rate)
        self.proj = layers.Dense(dim)

        num_window_elements = (2 * self.window_size[0] - 1) * (
            2 * self.window_size[1] - 1
        )
        self.relative_position_bias_table = self.add_weight(
            shape=(num_window_elements, self.num_heads),
            initializer=keras.initializers.Zeros(),
            trainable=True,
        )
        coords_h = np.arange(self.window_size[0])
        coords_w = np.arange(self.window_size[1])
        coords_matrix = np.meshgrid(coords_h, coords_w, indexing="ij")
        coords = np.stack(coords_matrix)
        coords_flatten = coords.reshape(2, -1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.transpose([1, 2, 0])
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)

        self.relative_position_index = tf.Variable(
            initial_value=relative_position_index,
            trainable=False,
            dtype=tf.int32,
        #self.relative_position_index = tf.Variable(
         #   initializer=relative_position_index,
         #   shape=relative_position_index.shape,
         #   dtype="int",
         #   trainable=False,
        )

    def call(self, x, mask=None):
        _, size, channels = x.shape
        head_dim = channels // self.num_heads
        x_qkv = self.qkv(x)
        x_qkv = tf.reshape(x_qkv, (-1, size, 3, self.num_heads, head_dim))
        x_qkv = tf.transpose(x_qkv, (2, 0, 3, 1, 4))
        q, k, v = x_qkv[0], x_qkv[1], x_qkv[2]
        q = q * self.scale
        k = tf.transpose(k, (0, 1, 3, 2))
        attn = q @ k

        num_window_elements = self.window_size[0] * self.window_size[1]
        relative_position_index_flat = tf.reshape(self.relative_position_index, (-1,))
        relative_position_bias = tf.gather(
            self.relative_position_bias_table, 
            relative_position_index_flat, 
            axis=0)

        relative_position_bias = tf.reshape(
            relative_position_bias,
            (num_window_elements, num_window_elements, -1),
        )
        relative_position_bias = tf.transpose(relative_position_bias, (2, 0, 1))
        attn = attn + tf.expand_dims(relative_position_bias, axis=0)

        if mask is not None:
            nW = mask.shape[0]
            mask_float = tf.cast(
                tf.expand_dims(tf.expand_dims(mask, axis=1), axis=0),
                "float32",
            )
            attn = tf.reshape(attn, (-1, nW, self.num_heads, size, size)) + mask_float
            attn = tf.reshape(attn, (-1, self.num_heads, size, size))
            attn = keras.activations.softmax(attn, axis=-1)
        else:
            attn = keras.activations.softmax(attn, axis=-1)
        attn = self.dropout(attn)

        x_qkv = attn @ v
        x_qkv = tf.transpose(x_qkv, (0, 2, 1, 3))
        x_qkv = tf.reshape(x_qkv, (-1, size, channels))
        x_qkv = self.proj(x_qkv)
        x_qkv = self.dropout(x_qkv)
        return x_qkv

In [ ]:
class SwinTransformer(layers.Layer):
    def __init__(
        self,
        dim,
        num_patches,
        num_heads,
        window_size= window_size,   #4,
        shift_size=0,
        num_mlp= num_mlp,   #384,  #512,  #1024,
        qkv_bias=True,
        dropout_rate=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.dim = dim  # number of input dimensions
        self.num_patches = num_patches  # number of embedded patches
        self.num_heads = num_heads  # number of attention heads

        # Ensure window_size is a tuple
        if isinstance(window_size, int):
            self.window_size = (window_size, window_size)
        else:
            self.window_size = window_size

        # Ensure shift_size is an integer or tuple
        if isinstance(shift_size, int):
            self.shift_size = (shift_size, shift_size)
        else:
            self.shift_size = shift_size

        self.num_mlp = num_mlp  # number of MLP nodes

        self.norm1 = layers.LayerNormalization(epsilon=1e-5)
        self.attn = WindowAttention(
            dim,
            window_size=self.window_size,  # Pass as a tuple directly
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            dropout_rate=dropout_rate,
        )
        self.drop_path = layers.Dropout(dropout_rate)
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)

        self.mlp = keras.Sequential(
            [
                layers.Dense(num_mlp),
                layers.Activation(keras.activations.gelu),
                layers.Dropout(dropout_rate),
                layers.Dense(dim),
                layers.Dropout(dropout_rate),
            ]
        )

        if min(self.num_patches) < min(self.window_size):  # Ensure comparison uses integers
            self.shift_size = (0, 0)
            self.window_size = (min(self.num_patches), min(self.num_patches))

    def build(self, input_shape):
        if self.shift_size == (0, 0):
            self.attn_mask = None
        else:
            height, width = self.num_patches
            h_slices = (
                slice(0, -self.window_size[0]),
                slice(-self.window_size[0], -self.shift_size[0]),
                slice(-self.shift_size[0], None),
            )
            w_slices = (
                slice(0, -self.window_size[1]),
                slice(-self.window_size[1], -self.shift_size[1]),
                slice(-self.shift_size[1], None),
            )
            mask_array = np.zeros((1, height, width, 1))
            count = 0
            for h in h_slices:
                for w in w_slices:
                    mask_array[:, h, w, :] = count
                    count += 1
            mask_array = tf.convert_to_tensor(mask_array)

            # Mask array to windows
            mask_windows = window_partition(mask_array, self.window_size[0])
            mask_windows = tf.reshape(
                mask_windows, [-1, self.window_size[0] * self.window_size[1]]
            )
            attn_mask = tf.expand_dims(mask_windows, axis=1) - tf.expand_dims(
                mask_windows, axis=2
            )
            attn_mask = tf.where(attn_mask != 0, -100.0, attn_mask)
            attn_mask = tf.where(attn_mask == 0, 0.0, attn_mask)
            self.attn_mask = tf.Variable(
                initial_value=attn_mask,
                shape=attn_mask.shape,
                dtype=attn_mask.dtype,
                trainable=False,
            )

    def call(self, x, training=False):
        height, width = self.num_patches
        _, num_patches_before, channels = x.shape
        x_skip = x
        x = self.norm1(x)
        x = tf.reshape(x, (-1, height, width, channels))
        if self.shift_size != (0, 0):
            shifted_x = tf.roll(
                x, shift=[-self.shift_size[0], -self.shift_size[1]], axis=[1, 2]
            )
        else:
            shifted_x = x

        x_windows = window_partition(shifted_x, self.window_size[0])
        x_windows = tf.reshape(
            x_windows, (-1, self.window_size[0] * self.window_size[1], channels)
        )
        attn_windows = self.attn(x_windows, mask=self.attn_mask)

        attn_windows = tf.reshape(
            attn_windows,
            (-1, self.window_size[0], self.window_size[1], channels),
        )
        shifted_x = window_reverse(
            attn_windows, self.window_size[0], height, width, channels
        )
        if self.shift_size != (0, 0):
            x = tf.roll(
                shifted_x, shift=[self.shift_size[0], self.shift_size[1]], axis=[1, 2]
            )
        else:
            x = shifted_x

        x = tf.reshape(x, (-1, height * width, channels))
        x = self.drop_path(x, training=training)
        x = x_skip + x
        x_skip = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = self.drop_path(x)
        x = x_skip + x
        return x

In [ ]:
# Apply patch extraction manually
def patch_extract(images, patch_size):
    batch_size = tf.shape(images)[0]
    patches = tf.image.extract_patches(
        images=images,
        sizes=(1, patch_size[0], patch_size[1], 1),
        strides=(1, patch_size[0], patch_size[1], 1),
        rates=(1, 1, 1, 1),
        padding="VALID",
    )
    patch_dim = patch_size[0] * patch_size[1] * images.shape[-1]
    num_patches = (images.shape[1] // patch_size[0]) * (images.shape[2] // patch_size[1])
    return tf.reshape(patches, (batch_size, num_patches, patch_dim))
    
# Patch Embedding
class PatchEmbedding(layers.Layer):
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.proj = layers.Dense(embed_dim)
        self.pos_embed = layers.Embedding(input_dim=num_patches, output_dim=embed_dim)

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        pos = tf.range(start=0, limit=self.num_patches, delta=1)
        pos = tf.broadcast_to(pos, [batch_size, self.num_patches])
        return self.proj(patch) + self.pos_embed(pos)

class PatchMerging(keras.layers.Layer):
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.embed_dim = embed_dim
        self.linear_trans = layers.Dense(2 * embed_dim, use_bias=False)
    def call(self, x):
        height, width = self.num_patches
        _, _, C = x.shape
        x = tf.reshape(x, (-1, height, width, C))
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = tf.concat((x0, x1, x2, x3), axis=-1)
        x = tf.reshape(x, (-1, (height // 2) * (width // 2), 4 * C))
        return self.linear_trans(x)

In [ ]:
##NEW
def augment(x):
    x = tf.image.random_crop(x, size=(image_dimension, image_dimension, 3))
    x = tf.image.random_flip_left_right(x)
    return x
dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(lambda x, y: (augment(x), y)) # Augmentation
    .batch(batch_size=batch_size)
    #.map(lambda x, y: (patch_extract(x, patch_size=(16, 16)), y)) 
    .prefetch(tf.data.experimental.AUTOTUNE)
    )
dataset_val = (
    tf.data.Dataset.from_tensor_slices((v_train, v_test))
    .batch(batch_size=batch_size)
    #.map(lambda x, y: (patch_extract(x, patch_size=(16, 16)), y)) # Pass `pat
    .prefetch(tf.data.experimental.AUTOTUNE)
    )
dataset_test = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(batch_size=batch_size)
    #.map(lambda x, y: (patch_extract(x, patch_size=(16, 16)), y)) # Pass `pat
    .prefetch(tf.data.experimental.AUTOTUNE)
    )

In [ ]:
def Swin_Block(input_shape=(128, 128, 3), patch_size=(16, 16)):                #def Swin_Block():
    patch_dim = patch_size[0] * patch_size[1] * input_shape[2]
    input_layer = layers.Input(shape=(64, patch_dim))

    # Patch Embedding
    x = PatchEmbedding(num_patches=num_patches, embed_dim=embed_dim)(input_layer)

    # Swin Transformer Block 1
    x = SwinTransformer(
        dim=embed_dim,
        num_patches=(num_patch_x, num_patch_y),
        num_heads=num_heads,
        window_size=window_size,
        shift_size=0,
        num_mlp=num_mlp,
        qkv_bias=qkv_bias,
        dropout_rate=dropout_rate,
    )(x)

    # Swin Transformer Block 2
    x = SwinTransformer(
        dim=embed_dim,
        num_patches=(num_patch_x, num_patch_y),
        num_heads=num_heads,
        window_size=window_size,
        shift_size=shift_size,
        num_mlp=num_mlp,
        qkv_bias=qkv_bias,
        dropout_rate=dropout_rate,
    )(x)

    # Patch Merging
    x = PatchMerging((num_patch_x, num_patch_y), embed_dim=embed_dim)(x)

    # Instead of pooling and Dense, return features:
    model = keras.Model(input_layer, x)
    return model

In [ ]:
# Step 1: Create the Swin encoder
swin_encoder = Swin_Block()

# Step 2: Create input tensor (image)
x_input = keras.Input(shape=(128, 128, 3))

# Step 3: Patchify the image first (match Swin input shape)
patches = Patches(patch_size)(x_input)  # shape: (batch, 64, 768)

# Step 4: Pass the patches to Swin encoder
swin_features = swin_encoder(patches)


In [ ]:
def CrossAttentionFusionBlock(vit_features, swin_features):
    # Cross-attention: ViT queries Swin
    cross_attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim,
        dropout=0.1
    )(query=vit_features, key=swin_features, value=swin_features)

    # Residual connection + LayerNorm
    x = layers.Add()([cross_attn, vit_features])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x

In [ ]:
# Custom Layer for Gated Fusion
class GatedDualAttentionFusion(layers.Layer):
    def __init__(self, projection_dim):
        super().__init__()
        self.concat = layers.Concatenate()
        self.dense_gate = layers.Dense(projection_dim, activation="sigmoid")
        self.subtract = layers.Subtract()
        self.multiply = layers.Multiply()
        self.add = layers.Add()
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, attn1, attn2):
        gate = self.dense_gate(self.concat([attn1, attn2]))
        gated1 = self.multiply([gate, attn1])
        gated2 = self.multiply([self.subtract([tf.ones_like(gate), gate]), attn2])
        fused = self.add([gated1, gated2])
        return self.norm(fused)

In [ ]:
patch_layer = Patches(patch_size=(16, 16))  # or whatever patch size you're using

# Main Model Definition
def CrossViT_Swin_Model():
    x_input = keras.Input(shape=(128, 128, 3))

    # Shared patch extraction
    patches = patch_layer(x_input)

    # ViT Encoder
    vit_encoder = ViT_Block()
    vit_features = vit_encoder(x_input)

    # Swin Encoder
    swin_encoder = Swin_Block()
    swin_features = swin_encoder(patches)
    swin_features_proj = layers.Dense(projection_dim)(swin_features)

    # Cross-Attention: ViT queries Swin
    fused = CrossAttentionFusionBlock(vit_features, swin_features_proj)

    # Gated Dual Attention Fusion (moved into a custom layer)
    attn1 = layers.MultiHeadAttention(num_heads=4, key_dim=projection_dim)(fused, fused)          ####
    attn2 = layers.MultiHeadAttention(num_heads=4, key_dim=projection_dim)(fused, swin_features_proj)     ####
    x = GatedDualAttentionFusion(projection_dim)(attn1, attn2)

    # Enhanced MLP Head
    for units in mlp_head_units:
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        x = layers.Dense(units, activation='gelu')(x)
        x = layers.Dropout(0.3)(x)

    
    # Final Self-Attention Layer
    x = layers.MultiHeadAttention(num_heads=8, key_dim=projection_dim)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    # Output Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)                #(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=x_input, outputs=outputs)
    return model

In [ ]:
# Compile the model
lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-4,
    first_decay_steps=10
)
optimizer = keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)          #weight_decay=1e-4)
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model = CrossViT_Swin_Model()
model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# # ⬇️ Insert GPU check here
# import tensorflow as tf
# tf.debugging.set_log_device_placement(True)
# dummy_input = tf.random.normal((2, 128, 128, 3))  # match your MRI shape
# with tf.device('/GPU:0'):
#     with tf.GradientTape() as tape:
#         preds = model(dummy_input, training=True)
#         loss = tf.reduce_mean(preds)
#     grads = tape.gradient(loss, model.trainable_variables)
# print("\n✅ GPU usage check complete — see above logs for any CPU fallbacks.")

history = model.fit(
    dataset,                     # Training dataset (tf.data.Dataset or (X_train, y_train))
    validation_data=dataset_val, # Validation dataset
    epochs=400                 # Number of epochs
)
loss, accuracy = model.evaluate(dataset_test)
print(f"Test loss: {round(loss, 2)}")
print(f"Test accuracy: {round(accuracy * 100, 2)}%")
#print(f"Test top 5 accuracy: {round(top_5_accuracy * 100, 2)}%")

In [ ]:
# ===================== TEACHER LIME =====================

from lime import lime_image
from skimage.segmentation import mark_boundaries
import numpy as np

np.random.seed(42)

teacher_model = model

def teacher_predict(images):
    images = np.array(images, dtype=np.float32)
    return teacher_model.predict(images, verbose=0)

explainer = lime_image.LimeImageExplainer()

# Select SAME sample for both teacher and student
idx = 10
sample_image = x_test[idx]

# Generate explanation
explanation_teacher = explainer.explain_instance(
    sample_image,
    teacher_predict,
    top_labels=1,
    num_samples=1000
)

# Extract regions
temp_t, mask_t = explanation_teacher.get_image_and_mask(
    explanation_teacher.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)

In [ ]:

np.random.seed(42)

# Prediction function
def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

# Initialize explainer
explainer = lime_image.LimeImageExplainer()

# Convert dataset → numpy
x_test_np = []
y_test_np = []

for x_batch, y_batch in dataset_test:
    x_test_np.append(x_batch.numpy())
    y_test_np.append(y_batch.numpy())

x_test_np = np.concatenate(x_test_np, axis=0)
y_test_np = np.concatenate(y_test_np, axis=0)

# -------- Single Sample --------
idx = 10
sample_image = x_test_np[idx]

explanation = explainer.explain_instance(
    image=sample_image,
    classifier_fn=predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(sample_image)
plt.title("Original MRI")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation")
plt.axis('off')

plt.tight_layout()
plt.savefig("lime_explanation.png", dpi=300)
plt.show()

# -------- Multiple Samples --------
for idx in [0, 5, 10]:
    sample_image = x_test_np[idx]

    explanation = explainer.explain_instance(
        sample_image,
        predict_fn,
        top_labels=1,
        num_samples=1000
    )

    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True,
        num_features=10,
        hide_rest=False
    )

    plt.imshow(mark_boundaries(temp, mask))
    plt.title(f"LIME - Sample {idx}")
    plt.axis('off')
    plt.show()

# -------- Comparison Plot (ONLY if attention_map exists) --------
# attention_map must be defined before this

# plt.figure(figsize=(15,4))
# plt.subplot(1,3,1)
# plt.imshow(sample_image)
# plt.title("Original")

# plt.subplot(1,3,2)
# plt.imshow(attention_map)
# plt.title("Attention")

# plt.subplot(1,3,3)
# plt.imshow(mark_boundaries(temp, mask))
# plt.title("LIME")

# plt.savefig("comparison_explainability.png", dpi=300)
# plt.show()

In [ ]:
input_shape = (128, 128, 3)
learning_rate = 0.001
weight_decay = 0.0001
batch_size = 16    #64   #32
image_size = 128 # We'll resize input images to this size
patch_size = 16 # Size of the patches to be extract from the input images
num_patches = (image_size // patch_size) ** 2
projection_dim = 32    #64
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
] # Size of the transformer layers
transformer_layers = 4  #2  #6  #8  #12  #16 #8
mlp_head_units = [1024, 512]    #[2048, 1024]        #[4096, 2048]     #[3072, 1536]    #[2048, 1024]   #[2048, 1024][128, 64]mlp_head_units = [3072, 1536]

data_augmentation = keras.Sequential(
    [
        #layers.Rescaling(1./255),       ####
        layers.Normalization(),
        #layers.Resizing(image_size, image_size),
        layers.RandomFlip("horizontal"),  # Updated for both horizontal and vertical flips   layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),        
        layers.RandomZoom(
            height_factor=0.2, width_factor=0.2
        ),
        #layers.RandomContrast(0.3),        ####
        #layers.RandomBrightness(0.2),      ####
        #layers.GaussianNoise(0.05),        ####
    ],
    name="data_augmentation",
)
# Compute the mean and the variance of the training data for normalization.
data_augmentation.layers[0].adapt(x_train)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x) # GELU activation func
        x = layers.Dropout(dropout_rate)(x) # Dropout layer
    return x

In [ ]:
# Student Model Block
def Student_Block():
    # Create the input tensor for the full model
    x_input = layers.Input(shape=input_shape)
    augmented = data_augmentation(x_input)
    patches = Patches(patch_size)(augmented)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    x = encoded_patches
    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attention_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        x = layers.Add()([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6)(x)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    logits = layers.Dense(3)(features)

    model = keras.Model(inputs=x_input, outputs=logits, name="student_transformer")
    return model

In [ ]:
student = Student_Block()
teacher = CrossViT_Swin_Model()

In [ ]:
import os

import keras
from keras import layers
from keras import ops
import numpy as np

In [ ]:
class Distiller(keras.Model):
    def __init__(self, student, teacher):
        super().__init__()
        self.teacher = teacher
        self.student = student

    def compile(
        self,
        optimizer,
        metrics,
        student_loss_fn,
        distillation_loss_fn,
        alpha=0.1,
        temperature=3,     
    ):
        """Configure the distiller.

        Args:
            optimizer: Keras optimizer for the student weights
            metrics: Keras metrics for evaluation
            student_loss_fn: Loss function of difference between student
                predictions and ground-truth
            distillation_loss_fn: Loss function of difference between soft
                student predictions and soft teacher predictions
            alpha: weight to student_loss_fn and 1-alpha to distillation_loss_fn
            temperature: Temperature for softening probability distributions.
                Larger temperature gives softer distributions.
        """
        super().compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature

    def compute_loss(
        self, x=None, y=None, y_pred=None, sample_weight=None, allow_empty=False
    ):
        teacher_pred = self.teacher(x, training=False)
        student_loss = self.student_loss_fn(y, y_pred)

        distillation_loss = self.distillation_loss_fn(
            ops.softmax(teacher_pred / self.temperature, axis=1),
            ops.softmax(y_pred / self.temperature, axis=1),
        ) * (self.temperature**2)

        loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss
        return loss

    def call(self, x):
        return self.student(x)

In [ ]:
distiller = Distiller(student=student, teacher=teacher)
distiller.compile(
    optimizer=tf.keras.optimizers.Adam(),
    metrics=[tf.keras.metrics.CategoricalAccuracy()],
    student_loss_fn=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    distillation_loss_fn=tf.keras.losses.KLDivergence(),
    alpha=0.9,  # 🟡 Important: reduce from 0.9
    temperature=9,  # 🟡 Important: reduce from 4.0
)

#history = distiller.fit(dataset, validation_data=dataset_val, epochs=20)
distiller.fit(x_train, y_train, epochs=400)    #400)

# # Evaluate student on test dataset
distiller.evaluate(x_test, y_test)

In [ ]:
# ===================== STUDENT LIME =====================

student_model = distiller.student

def student_predict(images):
    images = np.array(images, dtype=np.float32)
    return student_model.predict(images, verbose=0)

explanation_student = explainer.explain_instance(
    sample_image,
    student_predict,
    top_labels=1,
    num_samples=1000
)

temp_s, mask_s = explanation_student.get_image_and_mask(
    explanation_student.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)

# ===================== TEACHER vs STUDENT COMPARISON =====================

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(sample_image)
plt.title("Original MRI")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(mark_boundaries(temp_t, mask_t))
plt.title("Teacher LIME")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(mark_boundaries(temp_s, mask_s))
plt.title("Student LIME")
plt.axis('off')

plt.tight_layout()
plt.savefig("teacher_student_lime.png", dpi=300)
plt.show()

In [ ]:
# import tensorflow as tf
# import numpy as np
# import matplotlib.pyplot as plt
# import os

# # -------------------------------
# # Batched SHAP-style Gradient Attribution for MRI
# # -------------------------------

# def shap_style_attribution_batch(model, images):
#     """
#     Generates SHAP-like attributions for a batch of images.
#     model  : Trained TF/Keras model
#     images : NumPy array of shape (B, H, W, C)
#     Returns:
#         attributions: (B, H, W) array
#         pred_classes: (B,) array
#     """
#     img_tensor = tf.convert_to_tensor(images, dtype=tf.float32)

#     # Predict classes for all images
#     preds = model(img_tensor, training=False)
#     pred_classes = tf.argmax(preds, axis=1).numpy()

#     with tf.GradientTape() as tape:
#         tape.watch(img_tensor)
#         preds = model(img_tensor, training=False)
#         # Gather the predicted class scores for each sample
#         idx = tf.stack([tf.range(tf.shape(preds)[0]), pred_classes], axis=1)
#         target_scores = tf.gather_nd(preds, idx)

#     grads = tape.gradient(target_scores, img_tensor)

#     if grads is None:
#         H, W = images.shape[1:3]
#         return np.zeros((len(images), H, W)), pred_classes

#     # SHAP-like attribution: input * gradient
#     attributions = grads * img_tensor

#     # Convert to absolute values and normalize to [0, 1] per image
#     attributions = tf.reduce_mean(tf.abs(attributions), axis=-1)  # average over channels
#     a_min = tf.reduce_min(attributions, axis=(1, 2), keepdims=True)
#     a_max = tf.reduce_max(attributions, axis=(1, 2), keepdims=True)
#     attributions = (attributions - a_min) / (a_max - a_min + 1e-8)

#     return attributions.numpy(), pred_classes


# def plot_overlay(original_img, attribution_map, alpha=0.5, cmap='jet'):
#     """Overlays attribution heatmap on the original MRI slice."""
#     img_norm = (original_img - np.min(original_img)) / (np.max(original_img) - np.min(original_img) + 1e-8)
#     plt.imshow(img_norm, cmap='gray')
#     plt.imshow(attribution_map, cmap=cmap, alpha=alpha)
#     plt.axis('off')


# # -------------------------------
# # Run on all test samples in batches & save
# # -------------------------------

# trained_student = distiller.student
# output_dir = "shap_style_overlays"
# os.makedirs(output_dir, exist_ok=True)

# batch_size = 16  # Adjust based on GPU memory

# for start in range(0, len(x_test), batch_size):
#     end = min(start + batch_size, len(x_test))
#     batch_imgs = x_test[start:end]
#     batch_attribs, batch_preds = shap_style_attribution_batch(trained_student, batch_imgs)

#     for i, idx in enumerate(range(start, end)):
#         plt.figure(figsize=(4, 4))
#         plot_overlay(x_test[idx], batch_attribs[i])
#         plt.title(f"True: {np.argmax(y_test[idx])} | Pred: {batch_preds[i]}")
#         plt.savefig(
#             os.path.join(output_dir, f"sample_{idx}_true{np.argmax(y_test[idx])}_pred{batch_preds[i]}.png"),
#             bbox_inches='tight',
#             dpi=300
#         )
#         plt.close()

# print(f"✅ Saved {len(x_test)} SHAP-style overlays in '{output_dir}'")


In [ ]:
# Train student as doen usually
student.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.CategoricalAccuracy()],
)

# Train and evaluate student trained from scratch.
student.fit(x_train, y_train, epochs=200)    #200)
student.evaluate(x_test, y_test)

In [ ]:
# Create teacher and student
teacher_model = CrossViT_Swin_Model()
student_model = Student_Block()   # your distilled student

# Build models using dummy input
dummy = tf.random.normal((1, 128, 128, 3))
_ = teacher_model(dummy)
_ = student_model(dummy)

# PARAMETER COUNT
print("\n===== PARAMETER COUNT =====")
teacher_params = teacher_model.count_params()
student_params = student_model.count_params()

print(f"Teacher Parameters: {teacher_params:,}")
print(f"Student Parameters: {student_params:,}")

reduction = (1 - student_params / teacher_params) * 100
print(f"Parameter Reduction: {reduction:.2f}%")


# INFERENCE TIME (Student)
import time

# Warm-up
for _ in range(10):
    _ = student_model(dummy)

start = time.time()
for _ in range(100):
    _ = student_model(dummy)
end = time.time()

avg_time = (end - start) / 100
print(f"\nStudent Inference Time: {avg_time*1000:.2f} ms per image")


In [ ]:
import matplotlib.pyplot as plt
# summarize history for accuracy
plt.plot(history.history['accuracy'],label="train_acc")
plt.plot(history.history['val_accuracy'],label="val_acc")
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
#plt.savefig("/content/drive/MyDrive/Alzheimer disease Classification/accuracypl
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Training loss
plt.plot(history.history["loss"], label="Train Loss", linewidth=4)

# Validation loss
plt.plot(history.history["val_loss"], label="Validation Loss", linewidth=4)

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curve")
plt.legend()
plt.grid(True)

# Save the plot
plt.savefig("loss_curve.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(8, 6))
# plt.plot(history.history["loss"], label="Train Loss", color="blue", linewidth=2)
# plt.xlabel("Epochs")
# plt.ylabel("Loss")
# plt.title("Training Loss Over Epochs", fontsize=14)
# plt.legend()
# plt.grid(True)

# # Save the plot as an image
# plt.savefig("train_loss_curve.png", dpi=300, bbox_inches="tight")

# # Show the plot
# plt.show()

In [ ]:
y_pred_1=model.predict(dataset_test)

In [ ]:
y_pred=np.argmax(model.predict(dataset_test), axis=-1)

In [ ]:
y_test=np.argmax(y_test, axis=1)

In [ ]:
#Defining function for confusion matrix plot
def plot_confusion_matrix(y_test, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):

    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'
    #Compute confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

#print(cm)

    fig, ax = plt.subplots(figsize=(7,7))
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    # We want to show all ticks...
    ax.set(xticks=np.arange(cm.shape[1]),
            yticks=np.arange(cm.shape[0]),
            # ... and label them with the respective list entries
            xticklabels=classes, yticklabels=classes,
            title=title,
            ylabel='True label',
            xlabel='Predicted label')

    #Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
            rotation_mode="anchor")
    # Loop over data dimensions and create text annotations.
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax

np.set_printoptions(precision=4)

plt.figure(figsize=(12,12),dpi=300)    

In [ ]:
#Defining the class labels
class_names=['AD','CN','MCI']#,'GNS']#, 'SPS']#,'ANS']#,'kNS','SSS']
# Plotting non-normalized confusion matrix
plot_confusion_matrix(y_test, y_pred, classes = class_names, title='Confusion matrix')
#plt.savefig(r'D:\Pictures\3Journal_BE\Model_val\Test\cm.jpg')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.utils.multiclass import unique_labels

def plot_confusion_matrix(y_true, y_pred, classes, normalize=False, title=None, cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    
    # Show all ticks and label them
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')

    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    # Loop over data dimensions and create text annotations
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax

# Plot normalized confusion matrix
plot_confusion_matrix(y_test, y_pred, classes=class_names, normalize=True, title=None, cmap=plt.cm.Blues)
plt.show()

# Calculate and print metrics
results = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(results)
print('Accuracy Score:', accuracy_score(y_test, y_pred))
print('Classification Report:')
print(classification_report(y_test, y_pred))

# For model predictions (if needed)
# y_pred = np.argmax(model.predict(X_test_scaled), axis=-1)  # Use your test features here

In [ ]:
from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(y_test, y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Initialize lists to store sensitivity and specificity for each class
sensitivity_list = []
specificity_list = []

# Calculate sensitivity and specificity for each class
for i in range(cm.shape[0]):
    # True positive, true negative, false positive, false negative for class i
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    # Calculate sensitivity (true positive rate or recall)
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0.0
    # Calculate specificity (true negative rate)
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0

    # Append sensitivity and specificity to the respective lists
    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)

# Print sensitivity and specificity for each class
for i in range(len(sensitivity_list)):
    print(f"Class {i}: Sensitivity (True Positive Rate): {sensitivity_list[i]:.4f}, Specificity (True Negative Rate): {specificity_list[i]:.4f}")


In [ ]:
from sklearn.metrics import matthews_corrcoef
y_pred = y_pred > 0.5
matthews_corrcoef(y_test, y_pred)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
from itertools import cycle
import numpy as np

# Binarize the labels for multi-class ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
lw = 4

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_1[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
colors = cycle(['blue', 'black', 'green', 'red'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2.5,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=lw)
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
# plt.title('Receiver operating characteristic for multi-class')
plt.legend(loc="lower right")
plt.show()



In [ ]:
# import os
# import cv2
# import numpy as np
# import tensorflow as tf
# from tensorflow.keras.models import Model

# def generate_gradcam_heatmap(model, input_image, class_index, target_layer_name):
#     """
#     Generates a Grad-CAM heatmap for a single input image.
#     """
#     target_layer = model.get_layer(target_layer_name)
#     grad_model = Model([model.inputs], [target_layer.output, model.output])

#     with tf.GradientTape() as tape:
#         conv_outputs, predictions = grad_model(input_image)
#         loss = predictions[:, class_index]

#     grads = tape.gradient(loss, conv_outputs)
#     pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
#     conv_outputs = conv_outputs[0]
#     heatmap = tf.reduce_sum(tf.multiply(pooled_grads[:, tf.newaxis], conv_outputs), axis=-1)
#     heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
#     return heatmap.numpy()

In [ ]:
# def save_and_plot_gradcam(model_dict, x_test, y_test, output_dir="gradcam_outputs", img_size=(128, 128)):
#     os.makedirs(output_dir, exist_ok=True)
#     class_names = {0: "AD", 1: "CN", 2: "MCI"}
#     class_indices = {}

#     for i in range(len(y_test)):
#         label = np.argmax(y_test[i]) if isinstance(y_test[i], (list, np.ndarray)) else y_test[i]
#         if label not in class_indices:
#             class_indices[label] = i
#         if len(class_indices) == 3:
#             break

#     gradcam_outputs = {model_name: {} for model_name in model_dict}

#     for model_name, model in model_dict.items():
#         target_layer_name = [l.name for l in model.layers if 'multi_head_attention' in l.name][-1]

#         for class_id, idx in class_indices.items():
#             input_image = x_test[idx:idx+1]
#             heatmap = generate_gradcam_heatmap(model, input_image, class_id, target_layer_name)
#             heatmap = cv2.resize(heatmap, img_size)
#             heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)

#             original_image = input_image[0]
#             if np.max(original_image) <= 1:
#                 original_image = (original_image * 255).astype("uint8")
#             original_image = cv2.resize(original_image, img_size)

#             superimposed_img = cv2.addWeighted(original_image, 0.6, heatmap_color, 0.4, 0)
#             gradcam_outputs[model_name][class_id] = superimposed_img

#             fname = f"{output_dir}/{model_name}_gradcam_{class_names[class_id]}.png"
#             cv2.imwrite(fname, cv2.cvtColor(superimposed_img, cv2.COLOR_RGB2BGR))

#     return gradcam_outputs


In [ ]:
# import matplotlib.pyplot as plt

# def plot_gradcam_summary_grid(gradcam_outputs):
#     row_labels = ["Teacher", "Student"]
#     col_labels = ["AD", "CN", "MCI"]

#     fig, axs = plt.subplots(2, 3, figsize=(12, 6))
#     fig.suptitle("Grad-CAM Visualization", fontsize=20)

#     for row, model_name in enumerate(["teacher", "student"]):
#         for col, class_id in enumerate([0, 1, 2]):
#             axs[row, col].imshow(cv2.cvtColor(gradcam_outputs[model_name][class_id], cv2.COLOR_BGR2RGB))
#             axs[row, col].axis("off")
#             if row == 0:
#                 axs[row, col].set_title(col_labels[col], fontsize=14)
#             if col == 0:
#                 axs[row, col].set_ylabel(row_labels[row], fontsize=14)

#     plt.tight_layout(rect=[0, 0, 1, 0.95])
#     plt.savefig("gradcam_summary_grid.png", dpi=300)
#     plt.show()


In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity

# def compute_attention_consistency_score(folder="gradcam_outputs", classes=["AD", "CN", "MCI"]):
#     acs_scores = {}
#     for cls in classes:
#         teacher_path = f"{folder}/teacher_gradcam_{cls}.png"
#         student_path = f"{folder}/student_gradcam_{cls}.png"

#         teacher_cam = cv2.imread(teacher_path, cv2.IMREAD_GRAYSCALE)
#         student_cam = cv2.imread(student_path, cv2.IMREAD_GRAYSCALE)

#         if teacher_cam is None or student_cam is None:
#             print(f"Missing image for class: {cls}")
#             continue

#         if teacher_cam.shape != student_cam.shape:
#             student_cam = cv2.resize(student_cam, (teacher_cam.shape[1], teacher_cam.shape[0]))

#         t_flat = teacher_cam.flatten().astype(np.float32)
#         s_flat = student_cam.flatten().astype(np.float32)
#         t_flat /= (np.linalg.norm(t_flat) + 1e-8)
#         s_flat /= (np.linalg.norm(s_flat) + 1e-8)

#         acs = np.dot(t_flat, s_flat)
#         acs_scores[cls] = acs
#         print(f"ACS for {cls}: {acs:.4f}")

#     mean_acs = np.mean(list(acs_scores.values()))
#     print(f"\n📊 Average ACS: {mean_acs:.4f}")
#     return acs_scores, mean_acs


In [ ]:
# def plot_acs_scores(acs_scores, mean_acs):
#     plt.bar(acs_scores.keys(), acs_scores.values(), color='skyblue')
#     plt.axhline(mean_acs, linestyle='--', color='red', label=f"Mean ACS = {mean_acs:.4f}")
#     plt.title("Attention Consistency Score (Teacher vs Student)")
#     plt.ylabel("Cosine Similarity")
#     plt.xlabel("Class")
#     plt.legend()
#     plt.ylim(0, 1)
#     plt.grid(True, linestyle="--", alpha=0.5)
#     plt.tight_layout()
#     plt.show()


In [ ]:
# # Step 1: Generate Grad-CAM and save
# model_dict = {"teacher": teacher, "student": student}
# gradcam_outputs = save_and_plot_gradcam(model_dict, x_test, y_test)

# # Step 2: Plot grid
# plot_gradcam_summary_grid(gradcam_outputs)

# # Step 3: Compute and plot ACS
# acs_scores, mean_acs = compute_attention_consistency_score()
# plot_acs_scores(acs_scores, mean_acs)


In [ ]:
# for layer in teacher.layers:
#     if "multi_head_attention" in layer.name:
#         print(layer.name)

In [ ]:
# import os
# import tensorflow as tf
# import numpy as np
# import cv2
# from tensorflow.keras.models import Model

# # ------------ Grad-CAM Heatmap Generator ------------
# def generate_gradcam_heatmap(model, input_image, class_index, target_layer_name):
#     grad_model = Model([model.inputs], [model.get_layer(target_layer_name).output, model.output])
    
#     with tf.GradientTape() as tape:
#         conv_outputs, predictions = grad_model(input_image)
#         loss = predictions[:, class_index]
    
#     grads = tape.gradient(loss, conv_outputs)
#     pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
#     conv_outputs = conv_outputs[0]
    
#     heatmap = tf.reduce_sum(tf.multiply(pooled_grads[:, tf.newaxis], conv_outputs), axis=-1)
#     heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
#     return heatmap.numpy()

# # ------------ Overlay Heatmap ------------
# def overlay_gradcam(image, heatmap, alpha=0.4):
#     heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
#     heatmap = np.uint8(255 * heatmap)
#     heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
#     if np.max(image) <= 1:
#         image = (image * 255).astype("uint8")
#     superimposed = cv2.addWeighted(image, 1 - alpha, heatmap_color, alpha, 0)
#     return superimposed

# # ------------ Select N Test Images per Class ------------
# N = 3
# class_samples = {0: [], 1: [], 2: []}  # AD, CN, MCI

# for i, label in enumerate(y_test):
#     label_id = np.argmax(label)
#     if len(class_samples[label_id]) < N:
#         class_samples[label_id].append(i)
#     if all(len(v) == N for v in class_samples.values()):
#         break

# # ------------ Generate and Save Grad-CAM Images ------------
# for model_name, model in {"teacher": teacher, "student": student}.items():
#     target_layer_name = [layer.name for layer in model.layers if "multi_head_attention" in layer.name][-1]
    
#     for class_id, indices in class_samples.items():
#         for j, idx in enumerate(indices):
#             input_image = x_test[idx:idx+1]
#             heatmap = generate_gradcam_heatmap(model, input_image, class_id, target_layer_name)
#             overlay = overlay_gradcam(input_image[0], heatmap)
#             save_path = f"{model_name}_gradcam_{class_id}_{j}.png"
#             cv2.imwrite(save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

# # ------------ Compute ACS Score ------------
# def compute_acs_score(teacher_path, student_path):
#     t = cv2.imread(teacher_path, cv2.IMREAD_GRAYSCALE)
#     s = cv2.imread(student_path, cv2.IMREAD_GRAYSCALE)
#     if t is None or s is None:
#         return None
#     t = cv2.resize(t.astype(np.float32), (128, 128))
#     s = cv2.resize(s.astype(np.float32), (128, 128))
#     t /= (np.linalg.norm(t.flatten()) + 1e-8)
#     s /= (np.linalg.norm(s.flatten()) + 1e-8)
#     return np.dot(t.flatten(), s.flatten())

# # ------------ Per-Class ACS Scores ------------
# classes = {0: "AD", 1: "CN", 2: "MCI"}
# overall_scores = []

# for cls_id, cls_name in classes.items():
#     cls_scores = []
#     for j in range(len(class_samples[cls_id])):
#         t_path = f"teacher_gradcam_{cls_id}_{j}.png"
#         s_path = f"student_gradcam_{cls_id}_{j}.png"

#         if not os.path.exists(t_path) or not os.path.exists(s_path):
#             print(f"⚠️ Missing file for {cls_name} sample {j}")
#             continue
        
#         acs = compute_acs_score(t_path, s_path)
#         if acs is not None:
#             print(f"ACS for {cls_name} sample {j}: {acs:.4f}")
#             cls_scores.append(acs)
#             overall_scores.append(acs)
    
#     if cls_scores:
#         print(f"✅ Average ACS for {cls_name}: {np.mean(cls_scores):.4f}\n")
#     else:
#         print(f"❌ No ACS computed for class {cls_name}\n")

# if overall_scores:
#     print(f"📊 Overall Average ACS: {np.mean(overall_scores):.4f}")
# else:
#     print("❌ No ACS scores computed. Check Grad-CAM generation.")


In [ ]:
# # ------------ Generate and Save Grad-CAM Images ------------
# for model_name, model in {"teacher": teacher, "student": student}.items():
#     # Get the correct target attention layer name dynamically for each model
#     target_layer_name = [layer.name for layer in model.layers if "multi_head_attention" in layer.name][-1]

#     for class_id, idx in class_indices.items():
#         input_image = x_test[idx:idx+1]
#         heatmap = generate_gradcam_heatmap(model, input_image, class_id, target_layer_name)
#         overlay = overlay_gradcam(input_image[0], heatmap)
#         cv2.imwrite(f"{model_name}_gradcam_{class_id}.png", cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))


# # ------------ Compute ACS Score ------------
# def compute_acs_score(teacher_path, student_path):
#     t = cv2.imread(teacher_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
#     s = cv2.imread(student_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
#     t = cv2.resize(t, (128, 128))
#     s = cv2.resize(s, (128, 128))
#     t /= (np.linalg.norm(t.flatten()) + 1e-8)
#     s /= (np.linalg.norm(s.flatten()) + 1e-8)
#     return np.dot(t.flatten(), s.flatten())

# classes = {0: "AD", 1: "CN", 2: "MCI"}
# scores = []
# for cls_id, cls_name in classes.items():
#     acs = compute_acs_score(f"teacher_gradcam_{cls_id}.png", f"student_gradcam_{cls_id}.png")
#     print(f"ACS for {cls_name}: {acs:.4f}")
#     scores.append(acs)

# print(f"\n📊 Average ACS: {np.mean(scores):.4f}")

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import pandas as pd
# import tensorflow as tf
# import os

# def create_bland_altman_plots(
#     teacher_model, student_model, x_test, y_test,
#     class_names=['AD', 'CN', 'MCI'], save_dir="bland_altman_results"
# ):
#     """
#     Create and save Bland–Altman plots, return results dict
#     """
#     os.makedirs(save_dir, exist_ok=True)

#     # Get predictions
#     teacher_probs = teacher_model.predict(x_test)
#     student_probs = student_model.predict(x_test)

#     # Convert logits to probabilities
#     if np.any(teacher_probs < 0) or np.any(teacher_probs > 1):
#         teacher_probs = tf.nn.softmax(teacher_probs).numpy()
#     if np.any(student_probs < 0) or np.any(student_probs > 1):
#         student_probs = tf.nn.softmax(student_probs).numpy()

#     # Combined plot
#     fig, axes = plt.subplots(1, len(class_names), figsize=(6*len(class_names), 5))
#     fig.suptitle('Bland–Altman Plots: Teacher vs Student', fontsize=16)

#     results = {}

#     for class_idx, class_name in enumerate(class_names):
#         ax = axes[class_idx] if len(class_names) > 1 else axes

#         teacher_prob = teacher_probs[:, class_idx]
#         student_prob = student_probs[:, class_idx]
#         mean_prob = (teacher_prob + student_prob) / 2
#         diff_prob = teacher_prob - student_prob

#         mean_diff = np.mean(diff_prob)
#         std_diff = np.std(diff_prob)
#         upper_limit = mean_diff + 1.96 * std_diff
#         lower_limit = mean_diff - 1.96 * std_diff

#         # Scatter plot
#         ax.scatter(mean_prob, diff_prob, alpha=0.6, s=30)
#         ax.axhline(mean_diff, color='red', linestyle='-', linewidth=2,
#                    label=f'Mean diff: {mean_diff:.4f}')
#         ax.axhline(upper_limit, color='red', linestyle='--', linewidth=2,
#                    label=f'Upper LoA: {upper_limit:.4f}')
#         ax.axhline(lower_limit, color='red', linestyle='--', linewidth=2,
#                    label=f'Lower LoA: {lower_limit:.4f}')
#         ax.axhline(0, color='black', linestyle='-', alpha=0.3)

#         ax.set_xlabel('Mean Probability (Teacher + Student)/2')
#         ax.set_ylabel('Difference (Teacher - Student)')
#         ax.set_title(f'{class_name} Class')
#         ax.legend()
#         ax.grid(True, alpha=0.3)

#         # Save per-class plot
#         per_class_path = os.path.join(save_dir, f"bland_altman_{class_name}.png")
#         plt.figure(figsize=(5,4))
#         plt.scatter(mean_prob, diff_prob, alpha=0.6, s=30)
#         plt.axhline(mean_diff, color='red', linestyle='-', linewidth=2)
#         plt.axhline(upper_limit, color='red', linestyle='--', linewidth=2)
#         plt.axhline(lower_limit, color='red', linestyle='--', linewidth=2)
#         plt.axhline(0, color='black', linestyle='-', alpha=0.3)
#         plt.xlabel('Mean Probability')
#         plt.ylabel('Difference')
#         plt.title(f'Bland–Altman: {class_name}')
#         plt.grid(True, alpha=0.3)
#         plt.tight_layout()
#         plt.savefig(per_class_path, dpi=300)
#         plt.close()

#         results[class_name] = {
#             'mean_difference': mean_diff,
#             'std_difference': std_diff,
#             'upper_limit': upper_limit,
#             'lower_limit': lower_limit,
#             'within_limits_pct': np.mean((diff_prob >= lower_limit) & (diff_prob <= upper_limit)) * 100
#         }

#     # Save combined plot
#     combined_path = os.path.join(save_dir, "bland_altman_all_classes.png")
#     plt.tight_layout()
#     plt.savefig(combined_path, dpi=300)
#     plt.close(fig)

#     return results


# def save_bland_altman_summary(results, save_dir="bland_altman_results"):
#     os.makedirs(save_dir, exist_ok=True)
#     summary_path = os.path.join(save_dir, "bland_altman_summary.csv")
#     df = pd.DataFrame.from_dict(results, orient="index")
#     df.to_csv(summary_path)
#     print(f"[✔] Summary saved to {summary_path}")


# def detailed_sample_analysis(
#     teacher_model, student_model, x_test, y_test,
#     class_names=['AD', 'CN', 'MCI'], save_dir="bland_altman_results"
# ):
#     os.makedirs(save_dir, exist_ok=True)

#     teacher_probs = teacher_model.predict(x_test)
#     student_probs = student_model.predict(x_test)

#     if np.any(teacher_probs < 0) or np.any(teacher_probs > 1):
#         teacher_probs = tf.nn.softmax(teacher_probs).numpy()
#     if np.any(student_probs < 0) or np.any(student_probs > 1):
#         student_probs = tf.nn.softmax(student_probs).numpy()

#     if y_test.ndim == 1:
#         true_labels = y_test
#     else:
#         true_labels = np.argmax(y_test, axis=1)

#     analysis_data = []
#     for i, (t_prob, s_prob, true_label) in enumerate(zip(teacher_probs, student_probs, true_labels)):
#         for class_idx, class_name in enumerate(class_names):
#             analysis_data.append({
#                 'sample_idx': i,
#                 'class': class_name,
#                 'true_label': class_names[true_label],
#                 'teacher_prob': t_prob[class_idx],
#                 'student_prob': s_prob[class_idx],
#                 'mean_prob': (t_prob[class_idx] + s_prob[class_idx]) / 2,
#                 'difference': t_prob[class_idx] - s_prob[class_idx],
#                 'abs_difference': abs(t_prob[class_idx] - s_prob[class_idx])
#             })

#     df = pd.DataFrame(analysis_data)
#     detailed_path = os.path.join(save_dir, "detailed_disagreements.csv")
#     df.to_csv(detailed_path, index=False)
#     print(f"[✔] Detailed disagreements saved to {detailed_path}")
#     return df


# # Main
# if __name__ == "__main__":
#     print("Performing Bland–Altman Analysis...")

#     results = create_bland_altman_plots(model, distiller.student, x_test, y_test)
#     save_bland_altman_summary(results)

#     detailed_df = detailed_sample_analysis(model, distiller.student, x_test, y_test)

#     print("\nTop disagreements per class:")
#     for cls in ['AD', 'CN', 'MCI']:
#         worst = detailed_df[detailed_df['class'] == cls].nlargest(5, 'abs_difference')
#         print(f"\n{cls} — Top 5 disagreements:")
#         print(worst[['sample_idx', 'teacher_prob', 'student_prob', 'difference']])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import os

# ===============================
# Bland–Altman Plot Function
# ===============================
def create_bland_altman_plots(
    teacher_model, student_model, x_test, y_test,
    class_names=['AD', 'CN', 'MCI'], save_dir="bland_altman_results"
):
    """
    Create and save Bland–Altman plots, return results dict
    """
    os.makedirs(save_dir, exist_ok=True)

    # Get predictions
    teacher_probs = teacher_model.predict(x_test)
    student_probs = student_model.predict(x_test)

    # Convert logits to probabilities
    if np.any(teacher_probs < 0) or np.any(teacher_probs > 1):
        teacher_probs = tf.nn.softmax(teacher_probs).numpy()
    if np.any(student_probs < 0) or np.any(student_probs > 1):
        student_probs = tf.nn.softmax(student_probs).numpy()

    # Combined plot
    fig, axes = plt.subplots(1, len(class_names), figsize=(6*len(class_names), 5))
    fig.suptitle('Bland–Altman Plots: Teacher vs Student', fontsize=16)

    results = {}

    for class_idx, class_name in enumerate(class_names):
        ax = axes[class_idx] if len(class_names) > 1 else axes

        teacher_prob = teacher_probs[:, class_idx]
        student_prob = student_probs[:, class_idx]
        mean_prob = (teacher_prob + student_prob) / 2
        diff_prob = teacher_prob - student_prob

        mean_diff = np.mean(diff_prob)
        std_diff = np.std(diff_prob)
        upper_limit = mean_diff + 1.96 * std_diff
        lower_limit = mean_diff - 1.96 * std_diff

        # Scatter plot
        ax.scatter(mean_prob, diff_prob, alpha=0.6, s=30)
        ax.axhline(mean_diff, color='red', linestyle='-', linewidth=2,
                   label=f'Mean diff: {mean_diff:.4f}')
        ax.axhline(upper_limit, color='red', linestyle='--', linewidth=2,
                   label=f'Upper LoA: {upper_limit:.4f}')
        ax.axhline(lower_limit, color='red', linestyle='--', linewidth=2,
                   label=f'Lower LoA: {lower_limit:.4f}')
        ax.axhline(0, color='black', linestyle='-', alpha=0.3)

        ax.set_xlabel('Mean Probability (Teacher + Student)/2')
        ax.set_ylabel('Difference (Teacher - Student)')
        ax.set_title(f'{class_name} Class')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Save per-class plot
        per_class_path = os.path.join(save_dir, f"bland_altman_{class_name}.png")
        plt.figure(figsize=(5,4))
        plt.scatter(mean_prob, diff_prob, alpha=0.6, s=30)
        plt.axhline(mean_diff, color='red', linestyle='-', linewidth=2)
        plt.axhline(upper_limit, color='red', linestyle='--', linewidth=2)
        plt.axhline(lower_limit, color='red', linestyle='--', linewidth=2)
        plt.axhline(0, color='black', linestyle='-', alpha=0.3)
        plt.xlabel('Mean Probability')
        plt.ylabel('Difference')
        plt.title(f'Bland–Altman: {class_name}')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(per_class_path, dpi=300)
        plt.close()

        results[class_name] = {
            'mean_difference': mean_diff,
            'std_difference': std_diff,
            'upper_limit': upper_limit,
            'lower_limit': lower_limit,
            'within_limits_pct': np.mean((diff_prob >= lower_limit) & (diff_prob <= upper_limit)) * 100
        }

    # Save combined plot
    combined_path = os.path.join(save_dir, "bland_altman_all_classes.png")
    plt.tight_layout()
    plt.savefig(combined_path, dpi=300)
    plt.close(fig)

    return results

# ===============================
# Save Summary CSV
# ===============================
def save_bland_altman_summary(results, save_dir="bland_altman_results"):
    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, "bland_altman_summary.csv")
    df = pd.DataFrame.from_dict(results, orient="index")
    df.to_csv(summary_path)
    print(f"[✔] Summary saved to {summary_path}")

# Alias for compatibility with older code
save_bland_altman_results = save_bland_altman_summary

# ===============================
# Print Summary to Console
# ===============================
def print_bland_altman_summary(results):
    """
    Print summary statistics for Bland–Altman analysis
    """
    print("\n" + "="*60)
    print("BLAND–ALTMAN ANALYSIS SUMMARY")
    print("="*60)
    
    for class_name, stats in results.items():
        print(f"\n{class_name} CLASS:")
        print(f"  Mean Difference (Bias): {stats['mean_difference']:.4f}")
        print(f"  Standard Deviation: {stats['std_difference']:.4f}")
        print(f"  Upper Limit of Agreement: {stats['upper_limit']:.4f}")
        print(f"  Lower Limit of Agreement: {stats['lower_limit']:.4f}")
        print(f"  Points within limits: {stats['within_limits_pct']:.1f}%")

        # Bias interpretation
        if abs(stats['mean_difference']) < 0.05:
            bias_interpretation = "Low bias - good agreement"
        elif abs(stats['mean_difference']) < 0.1:
            bias_interpretation = "Moderate bias"
        else:
            bias_interpretation = "High bias - poor agreement"

        # Agreement interpretation
        if stats['within_limits_pct'] >= 95:
            agreement_interpretation = "Excellent agreement (≥95% within limits)"
        elif stats['within_limits_pct'] >= 90:
            agreement_interpretation = "Good agreement (≥90% within limits)"
        else:
            agreement_interpretation = "Poor agreement (<90% within limits)"

        print(f"  Bias Assessment: {bias_interpretation}")
        print(f"  Agreement Assessment: {agreement_interpretation}")

# ===============================
# Detailed Sample Analysis
# ===============================
def detailed_sample_analysis(
    teacher_model, student_model, x_test, y_test,
    class_names=['AD', 'CN', 'MCI'], save_dir="bland_altman_results"
):
    os.makedirs(save_dir, exist_ok=True)

    teacher_probs = teacher_model.predict(x_test)
    student_probs = student_model.predict(x_test)

    if np.any(teacher_probs < 0) or np.any(teacher_probs > 1):
        teacher_probs = tf.nn.softmax(teacher_probs).numpy()
    if np.any(student_probs < 0) or np.any(student_probs > 1):
        student_probs = tf.nn.softmax(student_probs).numpy()

    if y_test.ndim == 1:
        true_labels = y_test
    else:
        true_labels = np.argmax(y_test, axis=1)

    analysis_data = []
    for i, (t_prob, s_prob, true_label) in enumerate(zip(teacher_probs, student_probs, true_labels)):
        for class_idx, class_name in enumerate(class_names):
            analysis_data.append({
                'sample_idx': i,
                'class': class_name,
                'true_label': class_names[true_label],
                'teacher_prob': t_prob[class_idx],
                'student_prob': s_prob[class_idx],
                'mean_prob': (t_prob[class_idx] + s_prob[class_idx]) / 2,
                'difference': t_prob[class_idx] - s_prob[class_idx],
                'abs_difference': abs(t_prob[class_idx] - s_prob[class_idx])
            })

    df = pd.DataFrame(analysis_data)
    detailed_path = os.path.join(save_dir, "detailed_disagreements.csv")
    df.to_csv(detailed_path, index=False)
    print(f"[✔] Detailed disagreements saved to {detailed_path}")
    return df

# ===============================
# MAIN: KD Effectiveness Analysis
# ===============================
if __name__ == "__main__":
    print("\n" + "="*80)
    print("KNOWLEDGE DISTILLATION EFFECTIVENESS ANALYSIS")
    print("="*80)

    # Distilled Student vs Teacher
    results = create_bland_altman_plots(model, distiller.student, x_test, y_test)
    print_bland_altman_summary(results)
    save_bland_altman_results(results)

    # Optional: correlation analysis (define separately if needed)
    # correlation_analysis(model, distiller.student, x_test, y_test)

    # Detailed disagreement table
    detailed_df = detailed_sample_analysis(model, distiller.student, x_test, y_test)

    print("\nTop disagreements per class:")
    for cls in ['AD', 'CN', 'MCI']:
        worst = detailed_df[detailed_df['class'] == cls].nlargest(5, 'abs_difference')
        print(f"\n{cls} — Top 5 disagreements:")
        print(worst[['sample_idx', 'teacher_prob', 'student_prob', 'difference']])

    # Compare with Student from Scratch (if available)
    if 'student' in locals():
        print("\n" + "="*60)
        print("COMPARISON: DISTILLED vs STUDENT FROM SCRATCH")
        print("="*60)
        
        results_scratch = create_bland_altman_plots(model, student, x_test, y_test)
        print_bland_altman_summary(results_scratch)
        save_bland_altman_results(results_scratch)
